In [2]:
import torch
import torch.nn.functional as F

In [3]:
# 2x2 회색조 이미지 3개
images = torch.tensor([
    [[0.1, 0.2],
     [0.3, 0.4]],

    [[0.8, 0.7],
     [0.6, 0.5]],

    [[0.2, 0.9],
     [0.1, 0.8]],
])

# 각 이미지를 4개의 feature를 가진 벡터로 펼친다: [3, 2, 2] -> [3, 4]
# (batch_size, height, width) -> (batch_size, num_features)
features = images.reshape(images.shape[0], -1)

print("Images shape:", images.shape)
print("Features:")
print(features)
print("Features shape:", features.shape)

Images shape: torch.Size([3, 2, 2])
Features:
tensor([[0.1000, 0.2000, 0.3000, 0.4000],
        [0.8000, 0.7000, 0.6000, 0.5000],
        [0.2000, 0.9000, 0.1000, 0.8000]])
Features shape: torch.Size([3, 4])


In [4]:
# 클래스 번호: 0=cat, 1=chicken, 2=dog
labels = torch.tensor([0, 1, 2])

# 클래스 번호를 one-hot vector로 변환
one_hot_labels = F.one_hot(
    labels,
    num_classes=3,
).to(torch.float32)

print("Integer labels:")
print(labels)

print("\nOne-hot labels:")
print(one_hot_labels)
print("One-hot shape:", one_hot_labels.shape)

Integer labels:
tensor([0, 1, 2])

One-hot labels:
tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
One-hot shape: torch.Size([3, 3])


In [5]:
num_inputs = 4
num_classes = 3

# 각 input feature와 각 output class를 연결한다.
# weights.shape = (num_inputs, num_classes)
weights = torch.randn(num_inputs, num_classes) * 0.01

# 클래스마다 bias가 하나씩 존재한다.
bias = torch.zeros(num_classes)

# features: (batch_size, 4)
# weights : (4, class_num)
# logits  : (batch_size, class_num)
# 
# 각 행은 하나의 이미지,
# 각 열은 해당 클래스의 가공되지 않은 점수(logit)다
logits = features @ weights + bias

print("Weights shape:", weights.shape)
print("Bias shape:", bias.shape)

print("\nLogits:")
print(logits)
print("Logits shape:", logits.shape)

Weights shape: torch.Size([4, 3])
Bias shape: torch.Size([3])

Logits:
tensor([[ 0.0007, -0.0052,  0.0106],
        [-0.0127, -0.0155,  0.0163],
        [ 0.0013, -0.0002,  0.0129]])
Logits shape: torch.Size([3, 3])


In [6]:
def stable_softmax(logits):
    
    # 각 데이터의 가장 큰 logit를 뺀다.
    # exp 계산에서 overflow가 발생하는 것을 막음.
    shifted_logits = (
        # 행별 최대값 찾아서 빼기
        # keepdim=True: [3, 1]        
        logits - logits.max(dim=-1, keepdim=True).values
    )
    
    # 지수로 올리기
    exponentials = shifted_logits.exp()
    
    # 마지막 차원인 class 방향으로 정규화
    return (
        exponentials 
        / exponentials.sum(dim=-1, keepdim=True)
    )
    
    
probabilities = stable_softmax(logits)

print("Probabilities:")
print(probabilities)

print("\nProbability sums:")
print(probabilities.sum(dim=-1))


Probabilities:
tensor([[0.3329, 0.3309, 0.3362],
        [0.3304, 0.3295, 0.3401],
        [0.3322, 0.3317, 0.3361]])

Probability sums:
tensor([1., 1., 1.])


In [7]:
# 직접 구현한 softmax를 PyTorch 구현과 비교
torch_probabilities = torch.softmax(
    logits,
    dim=-1,
)

assert torch.allclose(
    probabilities,
    torch_probabilities,
)

print("Manual softmax:")
print(probabilities)

print("\nPyTorch softmax:")
print(torch_probabilities)

Manual softmax:
tensor([[0.3329, 0.3309, 0.3362],
        [0.3304, 0.3295, 0.3401],
        [0.3322, 0.3317, 0.3361]])

PyTorch softmax:
tensor([[0.3329, 0.3309, 0.3362],
        [0.3304, 0.3295, 0.3401],
        [0.3322, 0.3317, 0.3361]])
